In [1]:
import torch
from torchsummary import summary
import numpy as np
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchvision.models import vgg16_bn, VGG16_BN_Weights

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <9A4710B9-0DA3-36BB-9129-645F282E64B2> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so
  Expected in:     <ECC148AF-20FF-3EEE-BC75-4DD3E7455393> /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
class Spine(nn.Module):
    def __init__(self, vgg):
        super(Spine, self).__init__()
        
        for i, child in enumerate(vgg.children()):
            for param in child.parameters():
                param.requires_grad = False
                
            if i == 0:
                layers_list = child
        
        self.conv = nn.Sequential(*layers_list[:-1])

    def forward(self, x):

        x = self.conv(x)

        return x
    

In [3]:
DETECTION_CLASSES = 6
# !!!! класс с индексом 0 - задний фон !!!!
class СlassifierHead(nn.Module):
    def __init__(self):
        super(СlassifierHead, self).__init__()

        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Linear(in_features = 8192, out_features = 2048, bias=True),
            nn.BatchNorm1d(2048, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 2048, out_features = 2048, bias=True),
            nn.BatchNorm1d(2048, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 2048, out_features = 1024, bias=True),
            nn.BatchNorm1d(1024, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 1024, out_features = 512, bias=True),
            nn.BatchNorm1d(512, momentum = 0.8),
            nn.LeakyReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features = 512, out_features = DETECTION_CLASSES + 1, bias=True),
            nn.Softmax(dim=0)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc(x)

        return x

In [4]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print(device)

model = Spine(vgg16_bn(weights = VGG16_BN_Weights.IMAGENET1K_V1)).to(device)
classifier = СlassifierHead()

cpu


In [5]:
summary(model, (3,640,640))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 640, 640]           1,792
       BatchNorm2d-2         [-1, 64, 640, 640]             128
              ReLU-3         [-1, 64, 640, 640]               0
            Conv2d-4         [-1, 64, 640, 640]          36,928
       BatchNorm2d-5         [-1, 64, 640, 640]             128
              ReLU-6         [-1, 64, 640, 640]               0
         MaxPool2d-7         [-1, 64, 320, 320]               0
            Conv2d-8        [-1, 128, 320, 320]          73,856
       BatchNorm2d-9        [-1, 128, 320, 320]             256
             ReLU-10        [-1, 128, 320, 320]               0
           Conv2d-11        [-1, 128, 320, 320]         147,584
      BatchNorm2d-12        [-1, 128, 320, 320]             256
             ReLU-13        [-1, 128, 320, 320]               0
        MaxPool2d-14        [-1, 128, 1

In [6]:
import imageio.v2 as imageio
PATH = "365.jpg"

image = imageio.imread(PATH)
image = image.swapaxes(0,2).swapaxes(1,2)/255.0
image = torch.unsqueeze(torch.from_numpy(image), dim=0)

In [7]:
sample = model(image.float())
sample.shape

torch.Size([1, 512, 40, 40])

In [8]:
def anchor_boxes(sample_size = (40,40), box_size = (4,4), stride = 4):
    '''
    returns all boxes by parameters y_top, x_left, y_bot, x_right
    box tensor will be: sample[BATCH, :, box[0]:box[2], box[1]:box[3]]
    '''
    for y in range(0, sample_size[0], stride):
        for x in range(0, sample_size[1], stride):
            yield torch.tensor([y,x,y+box_size[0],x+box_size[1]])

In [43]:
def push_foreground_to_buffer(classifier, sample, buffer):

    boxes = [torch.unsqueeze(el, dim=0) for el in anchor_boxes()]
    boxes = torch.cat(boxes, dim=0)

    for BATCH in range(sample.shape[0]):
        # складываем в один тензор все ancor_box от одного элемента из батча Spine
        # будем передавать эти тензоры по одному в классификатор
        
        ancors_batch = sample[BATCH, :, boxes[0,0]:boxes[0,2], boxes[0,1]:boxes[0,3]]
        ancors_batch = torch.unsqueeze(ancors_batch, dim=0)
        
        for i in range(1, len(boxes)):
            batch_element = sample[BATCH, :, boxes[i,0]:boxes[i,2], boxes[i,1]:boxes[i,3]]
            batch_element = torch.unsqueeze(batch_element, dim=0)
            
            ancors_batch = torch.cat((ancors_batch, batch_element), dim=0)


        # отправляем полученный батч в классификатор
        classifierHead_response = classifier(ancors_batch.detach())
        # от classifierHead_response ищем лосс и делаем step при обучении

        
        verdict = torch.argmax(classifierHead_response, dim=1)


        # значения всех боксов которые не фон
        boxes_features = ancors_batch[verdict!=0]
        # позиции всех боксов которые не фон
        positions = boxes[verdict!=0]
        # классы этих боксов
        class_ids = verdict[verdict!=0]

        # отправляем полученный батч


In [44]:
push_foreground_to_buffer(classifier, sample, None)

torch.Size([89, 512, 4, 4])
torch.Size([89])
torch.Size([89, 4])
